> **Cópia pública saneada.** Os dados de entrada não acompanham este repositório. Leia `docs/reprodutibilidade.md` e `docs/privacidade_e_dados.md` antes da execução. Notebooks de coleta dependem de rede; notebooks de tratamento escrevem somente em `data/`, que é ignorada pelo Git.

# Coleta de dados

Definição do que ele apresenta. Coletar pelo portal de dados abertos do BNDES; Quais bases.

## 1. Importação das bibliotecas

Neste bloco, importamos as bibliotecas básicas que serão usadas na coleta de dados.

A biblioteca `pathlib` será usada para organizar caminhos de pastas e arquivos.  
A biblioteca `json` será usada para ler arquivos de configuração.  
A biblioteca `pandas` será usada para trabalhar com tabelas.  
A biblioteca `requests` será usada para acessar páginas e APIs do Portal de Dados Abertos do BNDES.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import requests

## 2. Definição das pastas do projeto

Neste bloco, definimos os caminhos das principais pastas usadas no projeto.

A variável `PROJECT_ROOT` representa a pasta principal do projeto.  
A pasta `configs` guarda arquivos de configuração.  
A pasta `data/raw` será usada para armazenar os arquivos brutos baixados do BNDES.  
A pasta `data/interim` será usada para bases intermediárias.  
A pasta `data/processed` será usada para bases finais limpas.  
A pasta `outputs` será usada para tabelas, gráficos e relatórios.

Essa organização ajuda a separar dados originais, dados tratados e resultados analíticos.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CONFIG_DIR = PROJECT_ROOT / "configs"

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "results"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"
REPORTS_DIR = OUTPUT_DIR / "reports"

PROJECT_ROOT

## 3. Criação e verificação das pastas de trabalho

Neste bloco, garantimos que todas as pastas necessárias para o projeto existem.

As principais pastas são:

- `data/raw`: dados brutos baixados diretamente do BNDES;
- `data/interim`: dados intermediários após algum tratamento inicial;
- `data/processed`: bases finais limpas;
- `results/tables`: tabelas geradas pela análise;
- `results/figures`: gráficos;
- `results/reports`: relatórios e arquivos finais.

In [ ]:
pastas = [
    CONFIG_DIR,
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
]

for pasta in pastas:
    pasta.mkdir(parents=True, exist_ok=True)

[pasta.exists() for pasta in pastas]

## 4. Leitura do arquivo de fontes do BNDES

Neste bloco, carregamos o arquivo `fontes_bndes.json`, localizado na pasta `configs`.

Esse arquivo registra as principais fontes que serão usadas no projeto, como:

- Portal de Dados Abertos do BNDES;
- repositório de exemplos no GitHub;
- Operações de Financiamento;
- Desembolsos Mensais;
- Políticas Operacionais;
- Fontes de Recursos.

A ideia é manter as fontes documentadas em um arquivo separado, em vez de deixar os links espalhados pelo notebook.

In [ ]:
arquivo_fontes = CONFIG_DIR / "fontes_bndes.json"

with open(arquivo_fontes, encoding="utf-8") as f:
    fontes = json.load(f)

fontes

## 5. Organização das fontes em formato de tabela

Neste bloco, transformamos a lista de bases iniciais em um `DataFrame` do pandas.

Isso facilita a leitura das fontes, permitindo visualizar:

- nome da base;
- link no Portal de Dados Abertos do BNDES;
- uso previsto no projeto.

Essa tabela ainda não baixa os dados. Ela apenas organiza as fontes que serão consultadas.

In [ ]:
df_fontes = pd.DataFrame(fontes["bases_iniciais"])

df_fontes

## 6. Definição da API CKAN do Portal do BNDES

Neste bloco, definimos o endereço da API CKAN do Portal de Dados Abertos do BNDES.

O CKAN é a plataforma usada pelo BNDES para organizar seus dados abertos.  
Por meio da API, podemos consultar informações sobre os datasets, como:

- título da base;
- descrição;
- arquivos disponíveis;
- formatos dos arquivos;
- links de download;
- identificadores dos recursos.

Nesta etapa, ainda não vamos baixar os arquivos. Vamos apenas preparar o endereço da API.

In [ ]:
CKAN_API = fontes["portal_dados_abertos"].rstrip("/") + "/api/3/action"

CKAN_API

## 7. Função para consultar a API CKAN

Neste bloco, criamos uma função chamada `ckan_action`.

Essa função recebe o nome de uma ação da API CKAN e alguns parâmetros opcionais.  
Ela monta a consulta, acessa o Portal de Dados Abertos do BNDES e retorna o resultado em formato Python.

Por exemplo, depois podemos usar essa função para:

- buscar informações de um dataset;
- listar arquivos disponíveis;
- obter links de download;
- consultar metadados das bases.

A função também verifica se a consulta foi bem-sucedida. Caso a API retorne erro, o código interrompe a execução e mostra uma mensagem.

In [ ]:
def ckan_action(action, **params):
    url = f"{CKAN_API}/{action}"
    
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    
    data = response.json()
    
    if not data.get("success"):
        raise RuntimeError(f"Erro na API CKAN: {action}")
    
    return data["result"]

## 8. Consulta de metadados da base Operações de Financiamento

Neste bloco, usamos a função `ckan_action` para consultar os metadados da base `operacoes-financiamento`.

Metadados são informações sobre a base, como:

- título;
- descrição;
- arquivos disponíveis;
- formato dos arquivos;
- links de download;
- identificadores dos recursos.

Essa etapa é importante porque, antes de baixar qualquer dado, precisamos saber exatamente quais arquivos existem dentro do dataset.

In [ ]:
metadados_operacoes = ckan_action(
    "package_show",
    id="operacoes-financiamento"
)

metadados_operacoes.keys()

## 9. Visualização das informações principais da base

Neste bloco, extraímos algumas informações básicas da base `Operações de Financiamento`.

Vamos observar:

- o nome interno da base;
- o título;
- a quantidade de arquivos disponíveis;
- a descrição publicada pelo BNDES.

Isso ajuda a entender o conteúdo da base antes de trabalhar com os arquivos.

In [ ]:
print("Nome interno:", metadados_operacoes["name"])
print("Título:", metadados_operacoes["title"])
print("Número de recursos:", metadados_operacoes["num_resources"])

print("\nDescrição:")
print(metadados_operacoes["notes"])

## 10. Listagem dos recursos da base Operações de Financiamento

Neste bloco, transformamos a lista de recursos da base `Operações de Financiamento` em uma tabela.

No CKAN, cada dataset pode ter vários recursos.  
Um recurso pode ser um arquivo CSV, um PDF de dicionário de dados ou outro tipo de arquivo.

Vamos visualizar:

- nome do recurso;
- formato;
- identificador do recurso;
- link de download;
- se o recurso está disponível no datastore da API.

Essa etapa nos ajuda a decidir quais arquivos devem ser baixados para a análise.

In [ ]:
recursos_operacoes = pd.DataFrame(metadados_operacoes["resources"])

colunas_recursos = [
    "name",
    "format",
    "id",
    "url",
    "datastore_active"
]

recursos_operacoes[colunas_recursos]

## 11. Separação entre dados e documentação

A base Operações de Financiamento possui recursos de dois tipos.

Os arquivos CSV contêm os dados que serão usados na análise quantitativa.
Os arquivos PDF são os dicionários de dados, necessários para interpretar corretamente as variáveis.

Neste bloco, vamos separar os recursos em dois grupos: arquivos de dados e arquivos de documentação.

In [ ]:
recursos_csv_operacoes = recursos_operacoes[
    recursos_operacoes["format"].str.upper() == "CSV"
].copy()

recursos_pdf_operacoes = recursos_operacoes[
    recursos_operacoes["format"].str.upper() == "PDF"
].copy()

print("Arquivos CSV:", len(recursos_csv_operacoes))
print("Arquivos PDF:", len(recursos_pdf_operacoes))

display(recursos_csv_operacoes[colunas_recursos])
display(recursos_pdf_operacoes[colunas_recursos])

## 12. Catálogo de recursos das quatro bases iniciais

Até aqui, analisamos apenas a base Operações de Financiamento.

Agora vamos repetir o mesmo procedimento para as quatro bases iniciais do projeto. O objetivo deste bloco é criar um catálogo com todos os recursos disponíveis em cada base, identificando quais são arquivos CSV, PDF ou outros formatos.

Este bloco ainda não baixa os arquivos. Ele apenas consulta a API do portal de dados abertos do BNDES e organiza o inventário dos recursos disponíveis.

In [ ]:
catalogo_recursos = []

for base in fontes["bases_iniciais"]:
    nome_pacote = base["url"].rstrip("/").split("/")[-1]
    pacote = ckan_action("package_show", id=nome_pacote)

    for recurso in pacote["resources"]:
        catalogo_recursos.append({
            "base": base["nome"],
            "pacote": nome_pacote,
            "titulo_pacote": pacote["title"],
            "recurso_nome": recurso.get("name"),
            "format": recurso.get("format"),
            "id": recurso.get("id"),
            "url": recurso.get("url"),
            "datastore_active": recurso.get("datastore_active"),
            "last_modified": recurso.get("last_modified"),
        })

catalogo_recursos = pd.DataFrame(catalogo_recursos)

catalogo_recursos

## 13. Resumo do catálogo por base e formato

Depois de montar o catálogo completo dos recursos, vamos resumir quantos arquivos existem por base e por formato.

Esse resumo ajuda a verificar se a coleta inicial está coerente e mostra quais arquivos serão usados como dados e quais serão usados como documentação.

In [ ]:
resumo_catalogo = (
    catalogo_recursos
    .groupby(["base", "format"], dropna=False)
    .size()
    .reset_index(name="quantidade_recursos")
    .sort_values(["base", "format"])
)

resumo_catalogo

## 14. Separação geral entre bases de dados e documentação

Com o catálogo completo das quatro bases, vamos separar os recursos em dois grupos.

O primeiro grupo contém os arquivos CSV, que serão usados para análise quantitativa.
O segundo grupo contém os arquivos PDF, que serão usados como documentação para interpretar corretamente as variáveis.

Essa separação é importante porque os dois tipos de arquivo têm funções diferentes no projeto.

In [ ]:
catalogo_csv = catalogo_recursos[
    catalogo_recursos["format"].str.upper() == "CSV"
].copy()

catalogo_pdf = catalogo_recursos[
    catalogo_recursos["format"].str.upper() == "PDF"
].copy()

print("Total de arquivos CSV:", len(catalogo_csv))
print("Total de arquivos PDF:", len(catalogo_pdf))

display(catalogo_csv[["base", "recurso_nome", "id", "datastore_active", "last_modified"]])
display(catalogo_pdf[["base", "recurso_nome", "id", "datastore_active", "last_modified"]])

## 15. Criação de nomes padronizados para os arquivos

Antes de baixar os arquivos, vamos criar nomes padronizados para salvá-los no projeto.

Essa etapa evita nomes muito longos, acentos, espaços e caracteres especiais. Também facilita a organização posterior dos dados brutos na pasta `data/raw`.

Cada arquivo receberá um nome baseado na base de origem, no nome do recurso e no formato do arquivo.

In [ ]:
import re
import unicodedata


def limpar_nome_arquivo(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    texto = texto.strip("_")
    return texto


catalogo_recursos["nome_arquivo"] = (
    catalogo_recursos["base"].apply(limpar_nome_arquivo)
    + "__"
    + catalogo_recursos["recurso_nome"].apply(limpar_nome_arquivo)
    + "."
    + catalogo_recursos["format"].str.lower()
)

catalogo_recursos[["base", "recurso_nome", "format", "nome_arquivo"]]

## 16. Exportação do catálogo dos dicionários de dados

Neste bloco, organizamos todos os dicionários de dados em uma única planilha Excel.

O arquivo reúne a base de origem, o nome do recurso, o formato, o identificador do recurso no portal CKAN, o link original, a data de modificação e o nome padronizado que será usado no projeto.

Esse arquivo servirá como índice metodológico dos dicionários das bases do BNDES.

In [ ]:
OUTPUT_REPORTS_DIR = PROJECT_ROOT / "results" / "reports"
OUTPUT_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if "nome_arquivo" not in catalogo_recursos.columns:
    catalogo_recursos["nome_arquivo"] = (
        catalogo_recursos["base"].apply(limpar_nome_arquivo)
        + "__"
        + catalogo_recursos["recurso_nome"].apply(limpar_nome_arquivo)
        + "."
        + catalogo_recursos["format"].str.lower()
    )

catalogo_pdf = catalogo_recursos[
    catalogo_recursos["format"].str.upper() == "PDF"
].copy()

arquivo_dicionarios = OUTPUT_REPORTS_DIR / "dicionarios_bndes.xlsx"

dicionarios_bndes = catalogo_pdf[
    [
        "base",
        "pacote",
        "titulo_pacote",
        "recurso_nome",
        "format",
        "id",
        "url",
        "datastore_active",
        "last_modified",
        "nome_arquivo",
    ]
].copy()

dicionarios_bndes.to_excel(
    arquivo_dicionarios,
    index=False,
    sheet_name="Dicionarios"
)

arquivo_dicionarios

## 17. Síntese da etapa de inventário

Nesta etapa, identificamos as quatro bases iniciais do projeto, consultamos seus recursos no portal de dados abertos do BNDES e organizamos o catálogo dos arquivos disponíveis.

O inventário encontrou 11 recursos no total: 6 arquivos CSV, que serão usados como bases de dados, e 5 arquivos PDF, que serão usados como dicionários e documentação metodológica.

Também foi criado um arquivo Excel com o catálogo dos dicionários de dados, permitindo consultar em um único lugar os links, identificadores e nomes padronizados dos documentos.

In [ ]:
print("Bases iniciais mapeadas:", catalogo_recursos["base"].nunique())
print("Total de recursos identificados:", len(catalogo_recursos))
print("Arquivos CSV:", len(catalogo_csv))
print("Arquivos PDF:", len(catalogo_pdf))
print("Catálogo dos dicionários:", arquivo_dicionarios)